# Notebook 7: Approach 6 - Frequency-Based NER + LLM Clustering
---
## Strategy
This is a **production-grade entity-centric clustering** strategy. Instead of relying purely on semantic embeddings (which can be noisy), we:
1.  **Extract Patterns**: Identify which entities (People, Orgs, Locations) frequently appear together.
2.  **LLM Naming**: Use an LLM (Gemini) to interpret these patterns and generate professional topic names.
3.  **Entity Tagging**: Assign documents to these "pattern-driven" clusters with high precision.

## Flow
```
NER CSV -> Calculate 2-3 Entity Combinations
        -> Select Top N Combinations
        -> Prompt LLM (Gemini) for Cluster Names & Related Entities
        -> Tag All Documents using Combinations + Related Entities
```

## Benefits
- **Extremely Low Noise**: Documents are only clustered if they share specific entity patterns.
- **Human-Readable Labels**: Clusters are named intelligently (e.g., "Middle East Military Tensions" instead of "iran_israel_attack").
- **Explainable AI**: You can see exactly why a document was placed in a cluster (which entities matched).

In [ ]:
# Install dependencies
!pip install -q -U google-generativeai pandas tqdm
print("Ready")

In [ ]:
import pandas as pd, collections, itertools, json, re, os
from tqdm.notebook import tqdm
from google.colab import files, userdata
import google.generativeai as genai

print("Setting up environment...")
try:
    # Attempt to get key from Colab Secrets
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    genai.configure(api_key=GEMINI_API_KEY)
    print("✅ Gemini API Key found in Secrets.")
except:
    print("⚠️ Gemini API Key not found in Secrets. Please enter it below.")
    GEMINI_API_KEY = input("Enter Gemini API Key: ")
    genai.configure(api_key=GEMINI_API_KEY)

uploaded = files.upload()
filename = list(uploaded.keys())[0]

## 1. Process NER Data & Find Combinations

In [ ]:
df = pd.read_csv(filename)
df[['Persons', 'Organizations', 'Locations']] = df[['Persons', 'Organizations', 'Locations']].fillna("")

def get_entities(row):
    ents = []
    for col in ['Persons', 'Organizations', 'Locations']:
        if row[col]:
            ents.extend([p.strip() for p in str(row[col]).split(',') if p.strip()])
    return list(set(ents))

print("Analyzing entity combinations (2-3 entities per title)...")
combo_counts = collections.Counter()
for _, row in df.iterrows():
    ents = get_entities(row)
    for r in range(1, 4): # Check 1, 2, and 3 entity sets
        if len(ents) >= r:
            for combo in itertools.combinations(sorted(ents), r):
                combo_counts[combo] += 1

top_combos = sorted([{'entities': k, 'count': v} for k, v in combo_counts.items() if v >= 3], 
                    key=lambda x: x['count'], reverse=True)[:50]

print(f"Found {len(top_combos)} significant entity patterns.")
for c in top_combos[:10]:
    print(f"  {', '.join(c['entities'])}: {c['count']} articles")

## 2. LLM Naming Service (Gemini)

In [ ]:
model = genai.GenerativeModel('gemini-1.5-flash')
cluster_definitions = {}

print("Requesting intelligent names from Gemini...")
for item in tqdm(top_combos):
    combo = item['entities']
    prompt = f"""
    Identify a professional news cluster name for these entities appearing together in {item['count']} articles:
    Entities: {', '.join(combo)}
    
    Return ONLY a JSON object with:
    {{
        "topic_name": "Short, clear cluster name",
        "related_entities": ["list", "of", "5", "frequently", "associated", "entities"],
        "context": "Historical or current news context in 1 sentence"
    }}
    """
    try:
        response = model.generate_content(prompt)
        match = re.search(r'\{.*\}', response.text, re.DOTALL)
        if match:
            cluster_definitions[combo] = json.loads(match.group())
        else:
            cluster_definitions[combo] = {"topic_name": " / ".join(combo), "related_entities": [], "context": ""}
    except:
        cluster_definitions[combo] = {"topic_name": " / ".join(combo), "related_entities": [], "context": ""}

print("\n✅ Naming Complete. Sample Clusters:")
for k, v in list(cluster_definitions.items())[:5]:
    print(f"Pattern {k} -> Cluster: {v['topic_name']}")

## 3. Final Document Tagging

In [ ]:
def tag_doc(row):
    doc_ents = set(get_entities(row))
    best_topic, best_score = None, 0
    
    for combo, meta in cluster_definitions.items():
        if set(combo).issubset(doc_ents):
            # Score = combo size + overlap with related entities
            overlap = len(doc_ents & set(meta.get('related_entities', [])))
            score = len(combo) * 10 + overlap
            if score > best_score:
                best_score = score
                best_topic = meta['topic_name']
    return best_topic if best_topic else "Other"

print("Assigning documents to clusters...")
df['Cluster_Name'] = df.apply(tag_doc, axis=1)

print("\nTop 10 Clusters by Document Count:")
print(df['Cluster_Name'].value_counts().head(10))

df.to_csv("approach_6_clustered.csv", index=False)
files.download("approach_6_clustered.csv")